In [1]:
import sys
from torch.profiler import profile, ProfilerActivity, record_function

TDECOMP_PATH = '..'
if not TDECOMP_PATH in sys.path:
    sys.path.append(TDECOMP_PATH)

Берём SmallLM 

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "arnir0/Tiny-LLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,  use_fast=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)


/tdecomp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 12/12 [00:00<00:00, 321.83it/s, Materializing param=model.norm.weight]                            


In [3]:
# train_imdb.py
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline
)
from datasets import load_dataset
import torch
# from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np

# model_name = "Qwen/Qwen2-0.5B-Instruct"
model_name = 'arnir0/Tiny-LLM'
dataset_name = "imdb"
# output_dir = "./qwen2-0.5b-imdb-finetuned"
output_dir = './tiny-llm'
max_length = 512  # Maximum context length for each sample

# Use 4-bit quantization to drastically reduce memory usage
use_4bit = False
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

# LoRA configuration for Parameter-Efficient Fine-Tuning
lora_r = 64
lora_alpha = 16
lora_dropout = 0.1

print("Loading model and tokenizer...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
# Set padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization if enabled
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
if use_4bit:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=use_4bit,
        bnb_4bit_quant_type=bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=use_nested_quant,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",  # Automatically places layers on available GPUs
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cuda:0", #auto
        torch_dtype=torch.float32,
        trust_remote_code=True
    )

# # Option 1: Tiny Shakespeare (literary text)
# dataset = load_dataset("tiny_shakespeare", split="train[:5%]")  # First 5%

# Option 2: CNN Daily Mail (news summaries) - smaller subset
# dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:100]")

# Option 3: Wikitext (Wikipedia articles) - small subset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")

# Option 4: Twitter Complaints (short text)
# dataset = load_dataset("twitter_complaints", split="train[:200]")

# Option 5: AG News (news articles)
# dataset = load_dataset("ag_news", split="train[:100]")

print(f"Dataset size: {len(dataset)}")
print(f"Dataset features: {dataset.features}")

# Preprocess the dataset based on its structure
def preprocess_dataset(examples):
    """Extract text from different dataset formats"""
    if 'text' in examples:
        return {"text": examples["text"]}
    elif 'article' in examples:  # CNN Daily Mail
        return {"text": examples["article"]}
    elif 'content' in examples:  # Some datasets
        return {"text": examples["content"]}
    elif 'sentence' in examples:  # Some sentence datasets
        return {"text": examples["sentence"]}
    else:
        # Try to use the first string column
        for key, value in examples.items():
            if isinstance(value[0], str):
                return {"text": examples[key]}
        return {"text": [str(x) for x in examples[list(examples.keys())[0]]]}

# Apply preprocessing
dataset = dataset.map(preprocess_dataset, batched=True)

# Filter out empty texts
dataset = dataset.filter(lambda example: len(example["text"].strip()) > 0)

# Tokenization function
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=128,  # Reduced for tiny model
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Split dataset
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")


# This will dynamically pad the batches during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We are doing causal LM, not masked LM
)

# Load accuracy metric
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Shift labels and predictions for causal LM (next token prediction)
    # Predictions are for the next token, so we shift labels accordingly
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # Flatten the tokens and get predictions
    predictions = np.argmax(shift_logits, axis=-1).flatten()
    labels = shift_labels.flatten()
    
    # Calculate accuracy, ignoring padding tokens (where label = -100)
    mask = labels != -100
    predictions = predictions[mask]
    labels = labels[mask]
    return accuracy_metric.compute(predictions=predictions, references=labels)

# ----------------------------
# 7. Training Arguments
# ----------------------------
# Training arguments
num_train_epochs = 3
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 4 #used for accumulate n batches in one gradient step if we havent enough memory to send all batch on device
learning_rate = 2e-4
logging_steps = 10
save_steps = 500


training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    # logging_steps=logging_steps,
    logging_steps=5,
    save_steps=save_steps,
    eval_strategy="steps",
    eval_steps=5,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="mlflow",  # Disable external logging like Weights & Biases for simplicity
    fp16=False,  # Use mixed precision training
)

Loading model and tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 12/12 [00:00<00:00, 690.06it/s, Materializing param=model.norm.weight]                            


Dataset size: 1000
Dataset features: {'text': Value('string')}
Training samples: 517
Validation samples: 130


In [4]:
print(len(dataset))
print(len(train_dataset))
dataset["text"][2]

647
517


" The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large team of writers handled the script . The game 's opening theme was sung by May 'n . \n"

In [5]:
from tdecomp.grad_proj.tensorgrad.config import TensorGRaDConfig, DataConfig, OptimizerConfig
import tdecomp.matrix.functional as F 

2026-02-18 09:56:53.215165: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-18 09:56:53.301352: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-18 09:56:54.579022: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


ParallelTG, ULTG - это вспомогательные классы-фабрики, убирающие лишние настройки, чтобы было проще. 
Если понадобится более тонкая настройка -- смотрите TensorGRaDConfig и передавайте соответствующие поля (там есть **kwargs) 

In [6]:
from tdecomp.grad_proj.tensorgrad.prepared_tg import ParallelTG, ULTG
a, b = ParallelTG(model, 
                    F.cur, # svd_type можете зарегистрировать свои разложения матричные в модуле F, 
                    # а также просто `truncated_svd` и `randomized_svd`
                    (8, 0.5), # первый и второй ранги. может быть Number, тогда ранги ставятся одинкаовыми.
                    # 0 < float < 1 интерпретируется как доля параметров, int - непосредственно ранг
                                  n_train=len(train_dataset),
                                  batch_size=per_device_train_batch_size,
                                  scheduler='StepLR'
                                  )
    # compute_metrics=compute_metrics, # Uncomment for per-epoch metrics (slower)
a

TensorGRaD (
Parameter Group 0
    betas: (0.9, 0.999)
    correct_bias: True
    eps: 1e-06
    initial_lr: 0.0001
    lr: 0.0001
    weight_decay: 0.0

Parameter Group 1
    batch_size: 4
    betas: (0.9, 0.999)
    correct_bias: True
    dim: 2
    enforce_full_complex_precision: False
    epochs: 100
    eps: 1e-06
    galore_2d_proj_type: left
    initial_lr: 0.0001
    lambda_sparse: 0.05
    lr: 0.0001
    n_iter_max_tucker: 10
    optimizer_type: tensorgrad_sum
    proj_type: low_rank
    rank: 8
    reset_sparse_optimizer_states: False
    scale: 1.0
    scale_by_mask_ratio: True
    scheduler_T_max: 100
    second_proj_type: unstructured_sparse
    second_rank: 0.5
    second_scale: 1.0
    second_scale_by_mask_ratio: False
    second_sparse_ratio: 0.25
    second_sparse_type: topk
    sparse_ratio: 0.1
    sparse_type: topk
    svd_type: <function cur at 0x7f204d4cca40>
    training_samples: 517
    tucker_warm_restart: True
    type: galore
    update_proj_gap: 100
    upda

In [31]:
import importlib
from examples import experiment_utils
importlib.reload(experiment_utils)
from tdecomp.grad_proj.tensorgrad.prepared_tg import ParallelTG, ULTG

parallel_tg_optimizer = ParallelTG(model, #TODO check that 0.5 не попадает в sparse_type а попадает в second_rank и затем никуда 
                                    F.cur, # svd_type можете зарегистрировать свои разложения матричные в модуле F, 
                                    # а также просто `truncated_svd` и `randomized_svd`
                                    (8, 0.5), # первый и второй ранги. может быть Number, тогда ранги ставятся одинкаовыми.
                                    # 0 < float < 1 интерпретируется как доля параметров, int - непосредственно ранг
                                    n_train=len(train_dataset),
                                    batch_size=per_device_train_batch_size,
                                    scheduler='StepLR',
                                    update_proj_gap=10, #надо понимать, что считаем в обобщённых шагах (batch_size * grad_acum) семплов - 1 шаг
                                    )
trainer_parallel_tg = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    optimizers=parallel_tg_optimizer,
    callbacks=[experiment_utils.UpdateGapMLflowCallback(), experiment_utils.SystemMetricsCallback()]
    # compute_metrics=compute_metrics, # Uncomment for per-epoch metrics (slower)
)

#TODO FIX!

print("First param group - 1D params")
for i in range(len(parallel_tg_optimizer[0].param_groups[0]['params'])):
    print(parallel_tg_optimizer[0].param_groups[0]['params'][i].shape)
parallel_tg_optimizer[0]

First param group - 1D params
torch.Size([192])
torch.Size([192])
torch.Size([192])


TensorGRaD (
Parameter Group 0
    betas: (0.9, 0.999)
    correct_bias: True
    eps: 1e-06
    initial_lr: 0.0001
    lr: 0.0001
    weight_decay: 0.0

Parameter Group 1
    batch_size: 4
    betas: (0.9, 0.999)
    correct_bias: True
    dim: 2
    enforce_full_complex_precision: False
    epochs: 100
    eps: 1e-06
    galore_2d_proj_type: left
    initial_lr: 0.0001
    lambda_sparse: 0.05
    lr: 0.0001
    n_iter_max_tucker: 10
    optimizer_type: tensorgrad_sum
    proj_type: low_rank
    rank: 8
    reset_sparse_optimizer_states: False
    scale: 1.0
    scale_by_mask_ratio: True
    scheduler_T_max: 100
    second_proj_type: unstructured_sparse
    second_rank: 0.5
    second_scale: 1.0
    second_scale_by_mask_ratio: False
    second_sparse_ratio: 0.25
    second_sparse_type: topk
    sparse_ratio: 0.1
    sparse_type: topk
    svd_type: <function cur at 0x7f204d4cca40>
    training_samples: 517
    tucker_warm_restart: True
    type: galore
    update_proj_gap: 10
    updat

In [8]:
! timeout 2 bash -lc 'echo > /dev/tcp/172.19.0.1/5000' && echo OK || echo FAIL

OK


In [9]:
! timeout 2 bash -lc 'echo > /dev/tcp/localhost/5000' && echo OK || echo FAIL

bash: connect: Connection refused
bash: line 1: /dev/tcp/localhost/5000: Connection refused
FAIL


In [10]:
! timeout 2 bash -lc 'echo > /dev/tcp/mlflow-network/5000' && echo OK || echo FAIL

bash: line 1: mlflow-network: Temporary failure in name resolution
bash: line 1: /dev/tcp/mlflow-network/5000: Invalid argument
FAIL


In [11]:
import mlflow
import mlflow.pytorch
from mlflow.server import get_app_client
import os

mlflow.set_tracking_uri("http://172.19.0.1:5000")
mlflow.set_experiment("tensorgrad-tdecomp-experiment")
# client = get_app_client("basic-auth", "http://172.19.0.1:5000")
# client.create_user(username="minio", password="minio123")
# # mlflow.export

os.environ["AWS_ACCESS_KEY_ID"] = "minio"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio123"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://172.19.0.1:9000" #https://github.com/mlflow/mlflow/issues/2150

In [12]:
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1) # - every second! not step!

In [13]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [14]:
print(len(train_dataset))
len(train_dataset['input_ids'])
len(eval_dataset) / 4

517


32.5

### Experiment

In [ ]:
print("Starting training...")
with mlflow.start_run(run_name="parallel_tg"):
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
                #  with_stack=True, 
                record_shapes=True,
                #experimental_config=torch._C._profiler._ExperimentalConfig(verbose=True),
                ) as profiler:
        with record_function("Record Train!") as rec1:
            trainer_parallel_tg.train()

2026/02/18 08:23:46 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Starting training...
### Using Composite Projector Configuration ###
    => Swapping projectors to ensure smaller one is first
    => Sizes after swap: first=0.25, second=8.0
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x76792bd0ef00>
UnstructuredSparseProjector initialized with sparse_ratio=0.25, sparse_type=topk, scale_by_mask_ratio=False
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configuration ###
    => Swapping projectors to ensure smaller one is first
    => Sizes after swap: first=0.25, second=8.0
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x76792ba07ce0>
UnstructuredSparseProjector initialized with sparse_ratio=0.25, sparse_type=topk, scale_by_mask_ratio=False
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configurati

/tdecomp/.venv/lib/python3.12/site-packages/tensorly/backend/__init__.py:202: UserWarning: An output with one or more elements was resized since it had shape [8, 192], which does not match the required output shape [32000, 192]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  return getattr(
/tdecomp/.venv/lib/python3.12/site-packages/tensorly/backend/__init__.py:202: UserWarning: An output with one or more elements was resized since it had shape [8, 192], which does not match the required output shape [192, 192]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggere

Step,Training Loss,Validation Loss
5,4.728196,4.747273
10,4.743232,4.732563
15,4.717543,4.722069
20,4.746300,4.709497
25,4.716271,4.700262
30,4.615441,4.689214
35,4.725477,4.688379
40,4.645448,4.687487
45,4.675849,4.686470
50,4.742678,4.685574


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.48it/s]
2026/02/18 08:24:41 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/02/18 08:24:41 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run parallel_tg at: http://172.19.0.1:5000/#/experiments/59/runs/f9d818e3d9ef4b76a781d1df1506af76
🧪 View experiment at: http://172.19.0.1:5000/#/experiments/59


In [33]:
ultg = ULTG(model, #TODO check that 0.5 не попадает в sparse_type а попадает в second_rank и затем никуда 
                    F.cur, # svd_type можете зарегистрировать свои разложения матричные в модуле F, 
                    # а также просто `truncated_svd` и `randomized_svd`
                    (8, 0.5), # первый и второй ранги. может быть Number, тогда ранги ставятся одинкаовыми.
                    # 0 < float < 1 интерпретируется как доля параметров, int - непосредственно ранг
                                  n_train=len(train_dataset),
                                  batch_size=per_device_train_batch_size,
                                  scheduler='StepLR',
                                  update_proj_gap=10, #надо понимать, что считаем в обобщённых шагах (batch_size * grad_acum) семплов - 1 шаг
                                  )
trainer_ultg = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    optimizers=ultg,
    callbacks=[experiment_utils.UpdateGapMLflowCallback(), experiment_utils.SystemMetricsCallback()]
    # compute_metrics=compute_metrics, # Uncomment for per-epoch metrics (slower)
)
ultg

(TensorGRaD (
 Parameter Group 0
     betas: (0.9, 0.999)
     correct_bias: True
     eps: 1e-06
     initial_lr: 0.0001
     lr: 0.0001
     weight_decay: 0.0
 
 Parameter Group 1
     batch_size: 4
     betas: (0.9, 0.999)
     correct_bias: True
     dim: 2
     enforce_full_complex_precision: False
     epochs: 100
     eps: 1e-06
     galore_2d_proj_type: left
     initial_lr: 0.0001
     lambda_sparse: 0.05
     lr: 0.0001
     n_iter_max_tucker: 10
     optimizer_type: tensorgrad
     proj_type: unstructured_sparse
     rank: 8
     reset_sparse_optimizer_states: False
     scale: 1.0
     scale_by_mask_ratio: True
     scheduler_T_max: 100
     second_proj_type: low_rank
     second_rank: 0.5
     second_scale: 1.0
     second_scale_by_mask_ratio: False
     second_sparse_ratio: 0.25
     second_sparse_type: topk
     sparse_ratio: 0.1
     sparse_type: topk
     svd_type: <function cur at 0x7f204d4cca40>
     training_samples: 517
     tucker_warm_restart: True
     type: gal

In [ ]:
with mlflow.start_run(run_name="ultg"):
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
                #  with_stack=True, 
                record_shapes=True,
                #experimental_config=torch._C._profiler._ExperimentalConfig(verbose=True),
                ) as profiler:
        with record_function("Record Train!") as rec2:
            trainer_ultg.train()

2026/02/18 10:19:35 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/02/18 10:19:35 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


### Using Composite Projector Configuration ###
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x7f22da8bfdd0>
UnstructuredSparseProjector initialized with sparse_ratio=0.1, sparse_type=topk, scale_by_mask_ratio=True
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configuration ###
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x7f22da8bf380>
UnstructuredSparseProjector initialized with sparse_ratio=0.1, sparse_type=topk, scale_by_mask_ratio=True
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configuration ###
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x7f22da8bf470>
UnstructuredSparseProjector initialized with sparse_ratio=0.1, sparse_type=topk, scale_by_mask_ratio=

/tdecomp/.venv/lib/python3.12/site-packages/tensorly/backend/__init__.py:202: UserWarning: An output with one or more elements was resized since it had shape [96, 192], which does not match the required output shape [32000, 192]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  return getattr(
/tdecomp/.venv/lib/python3.12/site-packages/tensorly/backend/__init__.py:202: UserWarning: An output with one or more elements was resized since it had shape [96, 192], which does not match the required output shape [192, 192]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Trigge

Step,Training Loss,Validation Loss
5,5.683937,6.175082
10,6.698241,9.674218
15,12.290372,16.927759
20,19.412502,21.550549
25,22.951720,24.764557
30,26.756165,28.558071
35,28.909317,28.969994
40,29.434485,29.509501
45,30.253510,30.064554
50,30.255283,30.614828


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.26it/s]


In [ ]:
torch.cuda.device_count()

In [59]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
comment="ultg"
profiler.export_chrome_trace(f"profile_{timestamp}_{comment}.json")